# B03b: Envelope Correlation and Connectivity (Quick Version)

**Duration**: ~25 minutes  
**Prerequisites**: B03a, B01

> 💡 **Quick Version**: Imports pre-built functions from `src/envelope.py`. See the full tutorial for implementation details.


## Table of Contents

1. [Introduction](#section-1-introduction)
2. [Envelope Correlation](#section-2-correlation)
3. [Volume Conduction Artifact](#section-3-volume-conduction)
4. [Multi-Band Analysis](#section-4-multi-band)
5. [Hyperscanning Application](#section-5-hyperscanning)
6. [Exercises](#section-6-exercises)
7. [Summary](#summary)
8. [External Resources](#external-resources)
9. [Discussion Questions](#discussion-questions)


In [ ]:
# =============================================================================
# Imports
# =============================================================================

import sys
from pathlib import Path

import numpy as np
from numpy.typing import NDArray
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

src_path = Path.cwd().parent.parent.parent / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from envelope import (
    extract_envelope,
    compute_envelope_correlation,
    compute_envelope_statistics,
    smooth_envelope_gaussian,
)
from constants import EEG_BANDS, BAND_COLORS
from colors import COLORS

plt.rcParams["figure.dpi"] = 100
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["font.size"] = 11

np.random.seed(42)
print("Imports loaded ✓")


---

<a id="section-1-introduction"></a>
## 1. Introduction

**Envelope Correlation** measures amplitude coupling between two signals:

$$\text{CCorr} = \text{Pearson}(A_1(t), A_2(t))$$

- Range: −1 (anti-correlated) to +1 (perfectly coupled)
- Sensitive to volume conduction (zero-lag spurious correlations)
- Complements phase-based metrics: captures *when* both signals are simultaneously strong

In hyperscanning: high envelope correlation may reflect shared arousal, joint attention, or emotional synchrony.


In [ ]:
fs = 250
t = np.arange(0, 6.0, 1/fs)

# ── Correlated scenario: shared amplitude modulation ──────────────────────
shared_mod = 0.5 + 0.5 * np.sin(2 * np.pi * 0.4 * t)
sig1_corr = shared_mod * np.sin(2 * np.pi * 10 * t + 0.0) + 0.1 * np.random.randn(len(t))
sig2_corr = shared_mod * np.sin(2 * np.pi * 10 * t + 1.5) + 0.1 * np.random.randn(len(t))  # different phase!

# ── Uncorrelated scenario: independent amplitude modulations ──────────────
mod1 = 0.5 + 0.5 * np.sin(2 * np.pi * 0.3 * t)
mod2 = 0.5 + 0.5 * np.sin(2 * np.pi * 0.7 * t + 1.2)
sig1_indep = mod1 * np.sin(2 * np.pi * 10 * t) + 0.1 * np.random.randn(len(t))
sig2_indep = mod2 * np.sin(2 * np.pi * 10 * t + 1.5) + 0.1 * np.random.randn(len(t))

corr_coupled   = compute_envelope_correlation(sig1_corr,  sig2_corr,  fs, band=(8, 13))
corr_uncoupled = compute_envelope_correlation(sig1_indep, sig2_indep, fs, band=(8, 13))

env1_c = extract_envelope(sig1_corr,  fs, (8, 13))
env2_c = extract_envelope(sig2_corr,  fs, (8, 13))
env1_u = extract_envelope(sig1_indep, fs, (8, 13))
env2_u = extract_envelope(sig2_indep, fs, (8, 13))

fig, axes = plt.subplots(2, 2, figsize=(14, 6))
for row, (e1, e2, corr, label) in enumerate([
    (env1_c, env2_c, corr_coupled,   f"Correlated (CCorr = {corr_coupled:.2f})"),
    (env1_u, env2_u, corr_uncoupled, f"Uncorrelated (CCorr = {corr_uncoupled:.2f})"),
]):
    axes[row, 0].plot(t, e1, color=COLORS["signal_1"], lw=1.5, label="Brain 1")
    axes[row, 0].plot(t, e2, color=COLORS["signal_2"], lw=1.5, label="Brain 2", alpha=0.8)
    axes[row, 0].set_title(label, fontweight="bold")
    axes[row, 0].legend()
    axes[row, 0].grid(True, alpha=0.3)
    axes[row, 1].scatter(e1, e2, alpha=0.3, s=8, color=COLORS["high_sync"])
    axes[row, 1].set_xlabel("Brain 1 Envelope")
    axes[row, 1].set_ylabel("Brain 2 Envelope")
    axes[row, 1].set_title(f"Scatter (r = {corr:.2f})", fontweight="bold")
    axes[row, 1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---

<a id="section-3-volume-conduction"></a>
## 3. Volume Conduction Artifact

Volume conduction creates **spurious** envelope correlations even when there's no true connectivity.
Two sensors picking up the same source will have perfectly correlated envelopes.


In [ ]:
np.random.seed(101)
t_vc = np.arange(0, 5.0, 1/fs)

# One true source with amplitude modulation
source_mod = 0.5 + 0.5 * np.sin(2 * np.pi * 0.5 * t_vc)
source = source_mod * np.sin(2 * np.pi * 10 * t_vc)

# Two sensors: both pick up the same source + independent noise
sensor1 = 0.8 * source + 0.2 * np.random.randn(len(t_vc))
sensor2 = 0.6 * source + 0.2 * np.random.randn(len(t_vc))

spurious_corr = compute_envelope_correlation(sensor1, sensor2, fs, band=(8, 13))
print(f"Spurious envelope correlation (volume conduction): {spurious_corr:.3f}")
print(f"→ High correlation (r ≈ {spurious_corr:.2f}) despite NO true connectivity between sensors")
print(f"→ This is the volume conduction problem for amplitude metrics")
print(f"\nSolution: orthogonalization (remove shared component before computing envelopes)")


---

<a id="section-4-multi-band"></a>
## 4. Multi-Band Analysis

Envelope dynamics differ across EEG frequency bands — each band operates on different timescales.


In [ ]:
np.random.seed(42)
t_mb = np.arange(0, 8.0, 1/fs)

# Rich synthetic EEG
delta = 0.8 * np.sin(2 * np.pi * 2 * t_mb)
alpha = (0.3 + 0.7 * np.sin(2 * np.pi * 0.3 * t_mb)) * np.sin(2 * np.pi * 10 * t_mb)
beta  = (0.2 + 0.3 * np.random.randn(len(t_mb)).clip(-1, 1)) * np.sin(2 * np.pi * 20 * t_mb)
eeg   = delta + alpha + beta + 0.3 * np.random.randn(len(t_mb))

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
selected_bands = [("delta", (1, 4)), ("alpha", (8, 13)), ("beta", (13, 30))]
colors_list = [COLORS["signal_1"], COLORS["negative"], COLORS["signal_3"]]

for ax, (band_name, band_range), color in zip(axes, selected_bands, colors_list):
    env = extract_envelope(eeg, fs, band_range)
    env_smooth = smooth_envelope_gaussian(env, sigma_samples=int(0.2 * fs))
    ax.plot(t_mb, env, color=color, lw=0.8, alpha=0.4)
    ax.plot(t_mb, env_smooth, color=color, lw=2, label=f"{band_name} ({band_range[0]}–{band_range[1]} Hz)")
    ax.legend(loc="upper right")
    ax.grid(True, alpha=0.3)
    ax.set_ylabel("Amplitude")
axes[-1].set_xlabel("Time (s)")
plt.suptitle("Envelope Dynamics Across Frequency Bands", fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


---

<a id="section-5-hyperscanning"></a>
## 5. Hyperscanning Application

In hyperscanning, we compute inter-brain envelope correlations across multiple channel pairs.


In [ ]:
np.random.seed(99)
t_hs = np.arange(0, 10.0, 1/fs)
n_channels = 4  # simplified

# Simulate two brains with partial amplitude coupling
shared_mod = 0.5 + 0.5 * np.sin(2 * np.pi * 0.2 * t_hs)

brain1_signals = []
brain2_signals = []
for ch in range(n_channels):
    phase_carrier = 2 * np.pi * np.random.uniform(0, 1)
    coupling = [0.8, 0.4, 0.1, 0.05][ch]  # varying coupling strength
    
    mod1 = shared_mod + 0.3 * np.random.randn(len(t_hs))
    mod2 = coupling * shared_mod + (1 - coupling) * (0.5 + 0.3 * np.random.randn(len(t_hs)))
    mod2 = np.clip(mod2, 0.05, None)
    
    brain1_signals.append(mod1 * np.sin(2 * np.pi * 10 * t_hs + phase_carrier))
    brain2_signals.append(mod2 * np.sin(2 * np.pi * 10 * t_hs + phase_carrier + 1.2))

# Compute inter-brain CCorr matrix
corr_matrix = np.zeros((n_channels, n_channels))
for i in range(n_channels):
    for j in range(n_channels):
        corr_matrix[i, j] = compute_envelope_correlation(
            brain1_signals[i], brain2_signals[j], fs, band=(8, 13)
        )

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr_matrix, vmin=-1, vmax=1, cmap="RdBu_r")
plt.colorbar(im, ax=ax, label="Envelope Correlation")
ax.set_xlabel("Brain 2 Channel")
ax.set_ylabel("Brain 1 Channel")
ax.set_title("Inter-Brain Alpha Envelope Correlation\n(simplified 4×4)", fontweight="bold")
ch_labels = [f"Ch{i+1}" for i in range(n_channels)]
ax.set_xticks(range(n_channels)); ax.set_xticklabels(ch_labels)
ax.set_yticks(range(n_channels)); ax.set_yticklabels(ch_labels)
for i in range(n_channels):
    for j in range(n_channels):
        ax.text(j, i, f"{corr_matrix[i,j]:.2f}", ha="center", va="center", fontsize=9)
plt.tight_layout()
plt.show()


---

<a id="section-6-exercises"></a>
## 6. Exercises

### 🎯 Exercise 1: CCorr vs Phase

**Difficulty**: 🟡 Medium  
**Task**: Create two signals where the phase difference is random (PLV ≈ 0) but envelope correlation is high. Then verify that `compute_envelope_correlation()` gives high CCorr while PLV is low.

```python
# Hint: use random phase carrier per trial but shared amplitude modulation
```

### 🎯 Exercise 2: Orthogonalization Preview

**Difficulty**: 🔴 Hard  
**Task**: For the volume conduction example above (sensor1, sensor2 from the same source), implement naive orthogonalization: regress signal2 onto signal1 and use the residuals. How does CCorr change?

```python
# Hint: np.linalg.lstsq can compute the linear regression coefficient
# residual = sensor2 - coef * sensor1
```


---

<a id="summary"></a>
## 7. Summary

### Key Concepts

| Concept | Key Point |
|---------|-----------|
| **Envelope Correlation** | Pearson r between two amplitude envelopes |
| **Range** | −1 to +1 (like standard correlation) |
| **Volume conduction** | Creates spurious high correlations via shared source |
| **Orthogonalization** | Removes linear zero-lag component to reduce artifact |
| **Hyperscanning** | Inter-brain CCorr matrix captures amplitude coupling patterns |

### Functions from `src/envelope.py`

| Function | Purpose |
|----------|---------|
| `compute_envelope_correlation` | Pearson r between envelopes of two signals |
| `compute_envelope_statistics` | Mean, std, CV, median for an envelope |


---

<a id="external-resources"></a>
## 8. External Resources

### 📚 Scientific References

- **Hipp, J.F. et al. (2012)** - *Large-scale cortical correlation structure of spontaneous oscillatory activity* — Orthogonalization for envelope correlation
- **Brookes, M.J. et al. (2011)** - *Measuring functional connectivity using MEG* — Amplitude envelope correlation

### 🎧 NotebookLM Resources

*(NotebookLM links will be added once this notebook is finalized)*


---

<a id="discussion-questions"></a>
## 9. Discussion Questions

1. **Phase vs Amplitude coupling**: Give an experimental example where you'd expect high PLV but low CCorr. What about high CCorr but low PLV?

2. **Orthogonalization trade-off**: Orthogonalization removes zero-lag coupling. Does it also remove true long-range amplitude coupling? What assumptions does orthogonalization make?

3. **Temporal resolution**: Envelope correlation collapses across time. How would you compute *time-resolved* envelope correlation? What window length would you need?
